# 6. Two-Optimizer Adversarial cVAE + MMD Batch Correction

**Pipeline:**
1. Load cohort from `data/interim_PT/`
2. Generate dummy split (`Split_dummy`: 60% train / 20% test / 20% NaN)
3. Fit Adv2-cVAE on **train** embeddings only
4. Transform **both** train and test -> 128-dim batch-invariant latent
5. Store in `.obsm["FM_COMPASS_PT_embedding_Adv2_corrected"]`
6. Save to `data/processed_two_opt/`
7. Compare with KL-cVAE results from `data/processed/`

## 0. Environment Setup

In [ ]:
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
# Fix scanpy/anndata version conflict in Colab
!pip install -q "scanpy==1.10.4" "anndata==0.10.9" "zarr<3"
!pip install -q scib-metrics leidenalg igraph
!pip install -q --upgrade ipython ipykernel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY,
    SCVI_CORRECTED_KEY, SCVI_LATENT_DIM,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.split_utils import add_dummy_split, get_split_masks
from batchcor_rna_emb.batch_correction.scvi_corrector import (
    CVAEAdv2Config, CVAEAdv2Corrector,
)
from batchcor_rna_emb.metrics.batch_metrics import (
    compute_kbet, compute_ilisi, compute_asw_batch, compute_graph_connectivity,
)
from batchcor_rna_emb.metrics.bio_metrics import (
    compute_clisi, compute_silhouette_bio, compute_nmi, compute_ari,
    run_leiden_clustering,
)
from batchcor_rna_emb.metrics.aggregation import geometric_mean

set_logger(level="INFO")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Imports OK, device: {}", DEVICE)

## 1. Mount Drive & Define Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Source: PreTrainer embeddings
DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
# Destination: Adv2 batch-corrected results (separate from KL-cVAE)
DATA_PROCESSED_ADV  = Path('/content/drive/MyDrive/data/processed_two_opt')
DATA_PROCESSED_ADV.mkdir(parents=True, exist_ok=True)
# Old KL-cVAE results for comparison
DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')

SPLIT_COL = "Split_dummy"
ADV2_CORRECTED_KEY = "FM_COMPASS_PT_embedding_Adv2_corrected"

logger.info("Source (interim_PT): {}", [p.name for p in sorted(DATA_INTERIM_PT_DIR.glob('*.adata.zarr'))])
logger.info("Destination (processed_two_opt): {}", DATA_PROCESSED_ADV)
logger.info("KL-cVAE results (processed): {}", [p.name for p in sorted(DATA_PROCESSED_DIR.glob('*.adata.zarr'))])

## 2. Process All Cohorts

For each cohort:
1. Load from `interim_PT`
2. Add `Split_dummy` (60/20/20)
3. Fit Adv2-cVAE on train -> transform train + test
4. Store corrected embeddings in `.obsm`
5. Save to `processed_two_opt/`

In [ ]:
adv2_config = CVAEAdv2Config(
    latent_dim=128,
    hidden_dims=(512, 256),
    disc_hidden=(256, 128),
    n_epochs=150,
    batch_size=128,
    lr_enc=5e-4,
    lr_disc=2e-4,
    lambda_adv=1.0,
    lambda_mmd=0.5,
    disc_steps=3,
    warmup_fraction=0.3,
    normalize=True,
    grad_clip=1.0,
    loss_type="both",
    seed=SEED,
)

all_histories = {}

for cohort_name in COHORT_CANCER_CODE:
    src_path  = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    dest_path = DATA_PROCESSED_ADV  / f"{cohort_name}.adata.zarr"

    if not src_path.exists():
        logger.warning("Cohort not found in interim_PT, skipping: {}", cohort_name)
        continue

    if dest_path.exists():
        logger.info("Already in processed_two_opt, skipping: {}", cohort_name)
        continue

    logger.info("\n" + "=" * 70)
    logger.info("Processing: {}", cohort_name)
    logger.info("=" * 70)

    adata = load_cohort(src_path)
    logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)

    if COMPASS_PT_EMBEDDING_KEY not in adata.obsm:
        logger.error("Missing '{}' in obsm, skipping", COMPASS_PT_EMBEDDING_KEY)
        del adata
        continue

    adata = add_dummy_split(adata, col_name=SPLIT_COL, seed=SEED)
    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)

    logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
    logger.info("Split: train={}, test={}, NaN={}",
        train_mask.sum(), test_mask.sum(),
        adata.n_obs - train_mask.sum() - test_mask.sum(),
    )

    emb_all = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    batch_labels_all = adata.obs[BATCH_COL].astype(str).values

    corrector = CVAEAdv2Corrector(config=adv2_config)
    corrector.fit(
        X=emb_all[train_mask],
        batch_labels=batch_labels_all[train_mask],
    )
    all_histories[cohort_name] = corrector.history_

    corrected = corrector.transform(emb_all)
    adata.obsm[ADV2_CORRECTED_KEY] = corrected

    logger.info("Corrected embedding shape: {}, dtype: {}",
        corrected.shape, corrected.dtype,
    )
    assert corrected.shape == (adata.n_obs, 128)
    assert np.isfinite(corrected).all(), "NaN/Inf in corrected embeddings!"

    save_adata_zarr(adata, dest_path)
    logger.info("Saved to: {}", dest_path)

    del adata, corrected
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

logger.info("\nAll cohorts processed.")
logger.info("processed_two_opt: {}", [p.name for p in sorted(DATA_PROCESSED_ADV.glob('*.adata.zarr'))])

## 3. Training Loss Curves

In [ ]:
n_cohorts = len(all_histories)
if n_cohorts == 0:
    logger.warning("No histories to plot (all cohorts were skipped).")
else:
    fig, axes = plt.subplots(n_cohorts, 4, figsize=(22, 4 * n_cohorts))
    if n_cohorts == 1:
        axes = axes[np.newaxis, :]

    for row, (name, hist) in enumerate(all_histories.items()):
        epochs = range(1, len(hist.recon) + 1)

        axes[row, 0].plot(epochs, hist.recon, color='#2196F3', linewidth=1.5)
        axes[row, 0].set_title(f"{name} - Reconstruction (MSE)", fontsize=10)
        axes[row, 0].set_xlabel("Epoch")

        axes[row, 1].plot(epochs, hist.disc_accuracy, color='#F44336', linewidth=1.5)
        axes[row, 1].set_title(f"{name} - Disc Accuracy", fontsize=10)
        axes[row, 1].set_xlabel("Epoch"); axes[row, 1].set_ylim(0, 1)

        axes[row, 2].plot(epochs, hist.adv_enc, color='#4CAF50', linewidth=1.5, label="Enc Adv")
        axes[row, 2].plot(epochs, hist.adv_disc, color='#FF9800', linewidth=1.5, label="Disc CE")
        axes[row, 2].set_title(f"{name} - Adversarial Losses", fontsize=10)
        axes[row, 2].legend(); axes[row, 2].set_xlabel("Epoch")

        axes[row, 3].plot(epochs, hist.mmd, color='#9C27B0', linewidth=1.5)
        axes[row, 3].set_title(f"{name} - MMD Loss", fontsize=10)
        axes[row, 3].set_xlabel("Epoch")

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 4. Before / After UMAP Comparison

Compare: Raw vs KL-cVAE (from `data/processed/`) vs Adv2-cVAE (from `data/processed_two_opt/`)

In [ ]:
from matplotlib.lines import Line2D
from umap import UMAP


def plot_umap_panel(coords, labels, title, ax, palette=None):
    unique = sorted(labels.unique())
    if palette is None:
        palette = sns.color_palette("husl", n_colors=len(unique))
    cmap = dict(zip(unique, palette))
    colors = [cmap[l] for l in labels]
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.6,
               edgecolors="none", rasterized=True)
    ax.set_title(title, fontsize=10, fontweight="bold")
    if len(unique) <= 12:
        handles = [Line2D([0], [0], marker='o', color='w',
                          markerfacecolor=cmap[u], markersize=6, label=u)
                   for u in unique]
        ax.legend(handles=handles, fontsize=7, loc="best")
    ax.set_xticks([]); ax.set_yticks([])


for cohort_name in COHORT_CANCER_CODE:
    adv_path = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"
    kl_path  = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"

    if not adv_path.exists():
        continue

    adata_adv = load_cohort(adv_path)

    keys_to_plot = {}
    keys_to_plot["Raw"] = np.asarray(adata_adv.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)

    if kl_path.exists():
        adata_kl = load_cohort(kl_path)
        if SCVI_CORRECTED_KEY in adata_kl.obsm:
            keys_to_plot["KL-cVAE"] = np.asarray(adata_kl.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
        del adata_kl

    keys_to_plot["Adv2-cVAE"] = np.asarray(adata_adv.obsm[ADV2_CORRECTED_KEY], dtype=np.float32)

    n_cols = len(keys_to_plot)
    fig, axes = plt.subplots(2, n_cols, figsize=(6 * n_cols, 10))
    if n_cols == 1:
        axes = axes[:, np.newaxis]
    fig.suptitle(f"UMAP: {cohort_name}", fontsize=14, fontweight="bold", y=1.02)

    for j, (label, emb) in enumerate(keys_to_plot.items()):
        reducer = UMAP(n_components=2, random_state=SEED)
        umap_coords = reducer.fit_transform(emb)
        plot_umap_panel(umap_coords, adata_adv.obs[BATCH_COL].astype(str),
                       f"{label} (Batch)", axes[0, j])
        plot_umap_panel(umap_coords, adata_adv.obs[DIAGNOSIS_COL].astype(str),
                       f"{label} (Diagnosis)", axes[1, j])

    plt.tight_layout(); plt.show()
    del adata_adv

## 5. Quantitative Check - Silhouette Scores

In [ ]:
from sklearn.metrics import silhouette_score


def safe_sil(X, labels):
    if len(np.unique(labels)) < 2:
        return np.nan
    return silhouette_score(X, labels)


rows = []
for name in COHORT_CANCER_CODE:
    adv_path = DATA_PROCESSED_ADV / f"{name}.adata.zarr"
    kl_path  = DATA_PROCESSED_DIR / f"{name}.adata.zarr"
    if not adv_path.exists():
        continue

    adata_adv = load_cohort(adv_path)
    emb_raw  = np.asarray(adata_adv.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    emb_adv2 = np.asarray(adata_adv.obsm[ADV2_CORRECTED_KEY], dtype=np.float32)
    batch = adata_adv.obs[BATCH_COL].astype(str).values
    diag  = adata_adv.obs[DIAGNOSIS_COL].astype(str).values

    row = {
        "Cohort": name,
        "n_batch": len(np.unique(batch)),
        "Sil_Batch_Raw": safe_sil(emb_raw, batch),
        "Sil_Diag_Raw":  safe_sil(emb_raw, diag),
        "Sil_Batch_Adv2": safe_sil(emb_adv2, batch),
        "Sil_Diag_Adv2":  safe_sil(emb_adv2, diag),
    }

    if kl_path.exists():
        adata_kl = load_cohort(kl_path)
        if SCVI_CORRECTED_KEY in adata_kl.obsm:
            emb_kl = np.asarray(adata_kl.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
            row["Sil_Batch_KL"] = safe_sil(emb_kl, batch)
            row["Sil_Diag_KL"]  = safe_sil(emb_kl, diag)
        del adata_kl

    rows.append(row)
    del adata_adv

df_sil = pd.DataFrame(rows)
display(df_sil.round(4))

## 6. Full scIB Metrics

- **Batch mixing**: kBET, iLISI, ASW_batch, Graph Connectivity -> **AvgBATCH**
- **Bio conservation**: cLISI, Silhouette_bio, NMI, ARI -> **AvgBio**

In [ ]:
import anndata as ad


def compute_all_metrics(adata, emb_key, batch_col, bio_col):
    n_batch = adata.obs[batch_col].nunique()
    n_bio = adata.obs[bio_col].nunique()
    m = {}
    if n_batch >= 2:
        try: m["kBET"] = compute_kbet(adata, emb_key, batch_col)
        except: m["kBET"] = np.nan
        try: m["iLISI"] = compute_ilisi(adata, emb_key, batch_col)
        except: m["iLISI"] = np.nan
        try: m["ASW_batch"] = compute_asw_batch(adata, emb_key, batch_col)
        except: m["ASW_batch"] = np.nan
        try: m["graph_conn"] = compute_graph_connectivity(adata, batch_col, emb_key=emb_key)
        except: m["graph_conn"] = np.nan
    else:
        m["kBET"] = m["iLISI"] = m["ASW_batch"] = m["graph_conn"] = np.nan

    if n_bio >= 2:
        try: m["cLISI"] = compute_clisi(adata, emb_key, bio_col)
        except: m["cLISI"] = np.nan
        try: m["Sil_bio"] = compute_silhouette_bio(adata, emb_key, bio_col)
        except: m["Sil_bio"] = np.nan
        try:
            adata_c = run_leiden_clustering(adata.copy(), emb_key, resolution=1.0)
            m["NMI"] = compute_nmi(adata_c, bio_col)
            m["ARI"] = compute_ari(adata_c, bio_col)
        except:
            m["NMI"] = m["ARI"] = np.nan
    else:
        m["cLISI"] = m["Sil_bio"] = m["NMI"] = m["ARI"] = np.nan

    batch_vals = [m[k] for k in ["kBET","iLISI","ASW_batch","graph_conn"] if not np.isnan(m.get(k, np.nan))]
    bio_vals = [m[k] for k in ["cLISI","Sil_bio","NMI","ARI"] if not np.isnan(m.get(k, np.nan))]
    m["AvgBATCH"] = geometric_mean(batch_vals) if batch_vals else np.nan
    m["AvgBio"] = geometric_mean(bio_vals) if bio_vals else np.nan
    return m


all_metrics = {}
for cohort_name in COHORT_CANCER_CODE:
    adv_path = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"
    kl_path  = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"
    if not adv_path.exists():
        continue

    adata_adv = load_cohort(adv_path)

    # Raw
    tmp = adata_adv.copy()
    tmp.obsm["X_emb"] = np.asarray(tmp.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    logger.info("Metrics: {} / Raw", cohort_name)
    all_metrics[f"{cohort_name}_Raw"] = compute_all_metrics(tmp, "X_emb", BATCH_COL, DIAGNOSIS_COL)
    all_metrics[f"{cohort_name}_Raw"]["Cohort"] = cohort_name
    all_metrics[f"{cohort_name}_Raw"]["Method"] = "Raw"

    # KL-cVAE
    if kl_path.exists():
        adata_kl = load_cohort(kl_path)
        if SCVI_CORRECTED_KEY in adata_kl.obsm:
            tmp_kl = adata_kl.copy()
            tmp_kl.obsm["X_emb"] = np.asarray(tmp_kl.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
            logger.info("Metrics: {} / KL-cVAE", cohort_name)
            all_metrics[f"{cohort_name}_KL"] = compute_all_metrics(tmp_kl, "X_emb", BATCH_COL, DIAGNOSIS_COL)
            all_metrics[f"{cohort_name}_KL"]["Cohort"] = cohort_name
            all_metrics[f"{cohort_name}_KL"]["Method"] = "KL-cVAE"
        del adata_kl

    # Adv2
    tmp_adv = adata_adv.copy()
    tmp_adv.obsm["X_emb"] = np.asarray(tmp_adv.obsm[ADV2_CORRECTED_KEY], dtype=np.float32)
    logger.info("Metrics: {} / Adv2-cVAE", cohort_name)
    all_metrics[f"{cohort_name}_Adv2"] = compute_all_metrics(tmp_adv, "X_emb", BATCH_COL, DIAGNOSIS_COL)
    all_metrics[f"{cohort_name}_Adv2"]["Cohort"] = cohort_name
    all_metrics[f"{cohort_name}_Adv2"]["Method"] = "Adv2-cVAE"
    del adata_adv

df_metrics = pd.DataFrame(all_metrics).T
cols = ["Cohort","Method","kBET","iLISI","ASW_batch","graph_conn","AvgBATCH",
        "cLISI","Sil_bio","NMI","ARI","AvgBio"]
df_metrics = df_metrics[[c for c in cols if c in df_metrics.columns]]
display(df_metrics.round(4))

## 7. Method Comparison per Cohort

In [ ]:
for cohort_name in COHORT_CANCER_CODE:
    mask = df_metrics["Cohort"] == cohort_name
    if mask.sum() == 0:
        continue
    df_c = df_metrics[mask].set_index("Method").drop(columns=["Cohort"])
    print(f"\n{'='*60}")
    print(f"  {cohort_name}: Method Comparison")
    print(f"{'='*60}")
    display(df_c.round(4))
    for col in df_c.columns:
        vals = df_c[col].dropna()
        if len(vals) > 0:
            best = vals.idxmax()
            print(f"  Best {col}: {best} ({vals[best]:.4f})")

## 8. Cross-Cohort Analysis (fixes NaN problem)

Merge all cohorts using `cohort_name` as batch label -> all metrics computable.

In [ ]:
all_adatas = []
for cohort_name in COHORT_CANCER_CODE:
    adv_path = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"
    kl_path  = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"
    if not adv_path.exists():
        continue
    adata = load_cohort(adv_path)
    adata.obs["cohort_batch"] = cohort_name

    if kl_path.exists():
        adata_kl = load_cohort(kl_path)
        if SCVI_CORRECTED_KEY in adata_kl.obsm:
            adata.obsm[SCVI_CORRECTED_KEY] = np.asarray(adata_kl.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
        del adata_kl

    all_adatas.append(adata)

merged = ad.concat(all_adatas, join="outer")
logger.info("Merged: %d samples, %d cohort-batches",
            merged.n_obs, merged.obs["cohort_batch"].nunique())

cross_results = {}

merged.obsm["X_eval"] = np.asarray(merged.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
cross_results["Raw"] = compute_all_metrics(merged, "X_eval", "cohort_batch", DIAGNOSIS_COL)

if SCVI_CORRECTED_KEY in merged.obsm:
    merged.obsm["X_eval"] = np.asarray(merged.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
    cross_results["KL-cVAE"] = compute_all_metrics(merged, "X_eval", "cohort_batch", DIAGNOSIS_COL)

merged.obsm["X_eval"] = np.asarray(merged.obsm[ADV2_CORRECTED_KEY], dtype=np.float32)
cross_results["Adv2-cVAE"] = compute_all_metrics(merged, "X_eval", "cohort_batch", DIAGNOSIS_COL)

df_cross = pd.DataFrame(cross_results).T
print("\n" + "=" * 60)
print("  Cross-Cohort Comparison (all cohorts merged)")
print("=" * 60)
display(df_cross.round(4))
del merged, all_adatas

## 9. Discriminator Convergence Diagnostic

In [ ]:
for cohort_name, hist in all_histories.items():
    adv_path = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"
    if adv_path.exists():
        adata_tmp = load_cohort(adv_path)
        n_batches = adata_tmp.obs[BATCH_COL].nunique()
        del adata_tmp
    else:
        n_batches = 5

    random_bl = 1.0 / n_batches

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(hist.disc_accuracy, color='#F44336', linewidth=1.5, label="Disc Accuracy")
    ax.axhline(random_bl, ls="--", color="gray", alpha=0.7, label=f"Random ({random_bl:.2f})")
    ax.set_title(f"Disc Accuracy: {cohort_name} (n_batches={n_batches})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)
    plt.tight_layout(); plt.show()

    final = np.mean(hist.disc_accuracy[-10:])
    status = "CONVERGED" if abs(final - random_bl) < 0.15 else "NOT converged"
    print(f"  {cohort_name}: final disc_acc={final:.3f} -> {status}")

## 10. Summary

### Data flow:
```
data/interim_PT/*.adata.zarr       (input: 4224-dim PreTrainer embeddings)
        |
        v
data/processed_two_opt/*.adata.zarr (output: + Adv2_corrected 128-dim latent)

data/processed/*.adata.zarr         (KL-cVAE results from notebook 4, comparison only)
```